# Atalaya Data support routing: multi-agent system

A Router sends each Query to one Area (HR, tech or finance) or to Out of Scope, a Specialist answers from that Area's Knowledge Base, and a Judge rates the Answer. Everything is traced in Langfuse.

This notebook only calls functions from the `orchestration` package, so it cannot drift from the code. Run it top to bottom with the keys configured in `.env`.

## 1. Setup and imports

The imports come from the `orchestration` package, plus `json` and `pathlib` from the standard library. The notebook never reads the API keys itself: they live in `.env` and `Settings` is read once, inside `multi_agent_system.py` (ADR 0003), so every other module only receives the values it needs.

In [1]:
import json
from pathlib import Path

from orchestration.area import AREA_DESCRIPTIONS, Area
from orchestration.checks import load_test_queries, sweep_threshold
from orchestration.multi_agent_system import build_support_system
from orchestration.observability import (
    format_experiment_summary,
    run_validation_experiment,
    upload_test_queries,
)
from orchestration.specialists import NO_INFORMATION_MESSAGE, OUT_OF_SCOPE_MESSAGE
from orchestration.support_answer import format_support_answer

## 2. Document loading and vector stores

`build_support_system()` reads the Documents in `data/`, splits them into Sections and indexes one Knowledge Base per Area in a persistent Chroma store under `.chroma/` (cosine distance, one embedding model for indexing and for Queries). Indexing is idempotent: running it again does not duplicate Sections. The result bundles `answer_query`, one `retrieve` function per Area and the Langfuse client.

The cell below also retrieves the Sections of the HR Knowledge Base for one Query, with their Similarity (1 minus the cosine distance).

In [2]:
system = build_support_system()

In [3]:
for section in system.retrieve_by_area[Area.HR]("¿Cuántos días de vacaciones tengo al año?"):
    print(f"{section.similarity:.3f}  {section.title}")

0.736  Días de vacaciones según la antigüedad
0.597  Plazo para pedir vacaciones
0.465  Licencia por fallecimiento de un familiar
0.452  Licencia por enfermedad y certificado médico


## 3. Agent definition (Specialists)

There is one Specialist per Area. Each is a deterministic chain, not a free-roaming agent (ADR 0002): it retrieves the top Sections from its Area's Knowledge Base, keeps only those whose Similarity reaches the Similarity Threshold, and asks the chat model to answer from them. If no Section survives, the Specialist abstains without calling the model (Abstention).

The calibrated values are a Similarity Threshold of **0.325** and top-k **4** (ADR 0004). They are stated here instead of read from `Settings`, which belongs to the composition module. The calibration sweep in section 6 shows how they were chosen.

The Area descriptions below are the single source for both the Router prompt and the Specialist prompt.

In [4]:
for area, description in AREA_DESCRIPTIONS.items():
    print(f"{area.value}: {description}\n")
print("Fixed messages:")
print("-", OUT_OF_SCOPE_MESSAGE)
print("-", NO_INFORMATION_MESSAGE)

hr: Resolves employment matters for Atalaya Staff: onboarding (background checks, the first month, when equipment is handed over), security clearances (levels, applications, renewals, what happens when one is lost or the person leaves), confidentiality and ethics, leave and absences, work modality (client offices, remote work, on-call duty, schedules), development (performance reviews, mandatory training, when salaries are reviewed or raised) and leaving the company (resignation, notice period). For Clients, it resolves questions about their assigned consultant: who it is, their clearance and how to request a replacement.

tech: Resolves anything that requires acting on a system. For Clients: platform access (creating users, passwords, two-factor authentication, locked accounts), roles and permissions (including removing access for someone who left the organisation), data classification, connecting data sources, dashboards and exports, audit logs, deployment and maintenance windows, an

## 4. Orchestrator and routing

The Router is a structured-output chain that returns a Route: a short reasoning first, then a target (`hr`, `tech`, `finance` or `out_of_scope`). A LangGraph graph runs the Route node, a conditional edge on the target sends the Query to the matching Specialist, and Out of Scope Queries get a fixed message with no retrieval. `answer_query` runs the graph inside one Langfuse trace.

One routing demo: a Query about a corporate card belongs to finance, even though it could sound like equipment.

In [5]:
answer = system.answer_query('¿Cómo solicito una tarjeta corporativa para viajar?')
print(format_support_answer(answer, system.langfuse.get_trace_url(trace_id=answer.trace_id)))

Query: ¿Cómo solicito una tarjeta corporativa para viajar?
Route: finance
Reasoning: La solicitud de una tarjeta corporativa para viajar se relaciona con los asuntos financieros y de gastos de los empleados de Atalaya, por lo que corresponde al área de finanzas.
Sections used:
  - Quién recibe una tarjeta corporativa (similarity 0.772)
  - Gastos permitidos con la tarjeta (similarity 0.596)
  - Devolución y baja de la tarjeta (similarity 0.570)
  - Límite mensual de la tarjeta (similarity 0.567)
Answer: Para solicitar una tarjeta corporativa para viajar, debes completar el formulario TC-01, el cual debe ser firmado por la jefatura directa del empleado. En este formulario, se debe explicar el motivo y la frecuencia de uso prevista. Luego, el área de Tesorería evaluará la solicitud en un plazo de 5 días hábiles y comunicará la decisión por correo. 

La tarjeta se entregará personalmente al titular, previa firma de la política de uso, y llegará al domicilio de la sede a la que pertenece e

## 5. Tests and examples

Example Queries written for this notebook (not taken from the Test Queries). Each output shows the Route with its reasoning, the Sections used with their Similarity, the Answer, the `abstained` flag, the Quality Verdict when the Answer was judged, and the Langfuse trace link. Abstentions and Out of Scope replies are never judged, so they have no verdict.

### One grounded Answer per Area

In [6]:
answer = system.answer_query('¿Cuántos días de vacaciones me corresponden al año?')
print(format_support_answer(answer, system.langfuse.get_trace_url(trace_id=answer.trace_id)))

Query: ¿Cuántos días de vacaciones me corresponden al año?
Route: hr
Reasoning: La consulta se refiere a un tema relacionado con el empleo y las políticas de vacaciones, que son gestionadas por el área de recursos humanos (hr).
Sections used:
  - Días de vacaciones según la antigüedad (similarity 0.744)
  - Plazo para pedir vacaciones (similarity 0.603)
  - Licencia por fallecimiento de un familiar (similarity 0.489)
  - Licencia sin goce de haberes (similarity 0.456)
Answer: Los días de vacaciones anuales se calculan según la antigüedad que tienes al 31 de diciembre de cada año. Con menos de cinco años de antigüedad, te corresponden 15 días hábiles de vacaciones. Entre cinco y diez años, corresponden 20 días hábiles, y con más de diez años, 25 días hábiles. Durante el primer año de trabajo, los días se otorgan en proporción al tiempo trabajado: se genera un día hábil por cada mes completo de servicio, y pueden tomarse una vez superado el período de prueba. El saldo disponible aparece 

In [7]:
answer = system.answer_query('¿Cómo configuro la autenticación de dos factores en la plataforma?')
print(format_support_answer(answer, system.langfuse.get_trace_url(trace_id=answer.trace_id)))

Query: ¿Cómo configuro la autenticación de dos factores en la plataforma?
Route: tech
Reasoning: La autenticación de dos factores es un aspecto técnico relacionado con el acceso a la plataforma, por lo que la consulta se dirige al área de tecnología que maneja la configuración de usuarios y permisos.
Sections used:
  - Configurar el doble factor de autenticación (similarity 0.715)
  - Recuperar una contraseña olvidada (similarity 0.550)
  - Requisitos de las contraseñas de la plataforma (similarity 0.533)
  - Cuentas de conexión y rotación de credenciales (similarity 0.530)
Answer: Para configurar la autenticación de dos factores en la plataforma, debes seguir estos pasos:

1. Instala en tu teléfono una aplicación autenticadora que genere códigos de seis dígitos.
2. Escanea el código QR que muestra la plataforma en "Mi perfil", opción "Seguridad".
3. Ingresa el primer código generado para confirmar el vínculo.

Al completar este proceso, la plataforma te entregará ocho códigos de respa

In [8]:
answer = system.answer_query('¿Cómo puedo descargar una factura emitida a mi organismo?')
print(format_support_answer(answer, system.langfuse.get_trace_url(trace_id=answer.trace_id)))

Query: ¿Cómo puedo descargar una factura emitida a mi organismo?
Route: finance
Reasoning: La consulta se refiere a la descarga de una factura, lo cual está relacionado con temas de facturación y pagos. Por lo tanto, el área que resuelve esta solicitud es finanzas.
Sections used:
  - Dónde descargar las facturas (similarity 0.660)
  - Recibo oficial de pago (similarity 0.619)
  - Dónde ver y descargar el recibo (similarity 0.591)
  - Entrega de facturas por mesa de entradas (similarity 0.583)
Answer: Puedes descargar una factura emitida a tu organismo desde el Portal de Clientes, en la sección "Comprobantes" dentro de "Facturación", dentro de las 2 horas posteriores a su emisión. Cada comprobante está disponible en PDF y en formato estructurado XML. Además, se envía una copia en PDF al correo del responsable administrativo registrado en el contrato. Si necesitas más información sobre cómo solicitar comprobantes archivados, puedes escribir a facturacion@atalayadata.example indicando el 

### Frontier Query

This Query touches two Areas. The Router must pick the Area whose staff act to resolve it (ADR 0001).

In [9]:
answer = system.answer_query('Perdí mi tarjeta de acceso a las oficinas, ¿a quién se lo digo y qué pasa con mi autorización de seguridad?')
print(format_support_answer(answer, system.langfuse.get_trace_url(trace_id=answer.trace_id)))

Query: Perdí mi tarjeta de acceso a las oficinas, ¿a quién se lo digo y qué pasa con mi autorización de seguridad?
Route: hr
Reasoning: La consulta se refiere a la pérdida de una tarjeta de acceso y cómo afecta la autorización de seguridad, lo cual está relacionado con los procedimientos de seguridad interna de Atalaya. Por lo tanto, el área que resuelve este tipo de consultas es hr, ya que se ocupa de los asuntos de seguridad y autorizaciones para el personal.
Sections used:
  - Qué hacer si se pierde la credencial de habilitación (similarity 0.676)
  - Qué pasa con la habilitación de seguridad al irse (similarity 0.570)
  - Devolución de credenciales y baja de accesos (similarity 0.564)
  - Acceso por error a datos de un organismo que no correspondían (similarity 0.524)
Answer: Debes comunicar la pérdida de tu tarjeta de acceso de inmediato. Llama a la guardia interna de Seguridad de la Información al interno 5100 y confirma el hecho por correo electrónico a seguridad@atalayadata.exa

### Out of Scope Query

No Area resolves this Query, so no Specialist runs and the fixed message is returned.

In [10]:
answer = system.answer_query('¿Cuál es la mejor receta de paella valenciana?')
print(format_support_answer(answer, system.langfuse.get_trace_url(trace_id=answer.trace_id)))

Query: ¿Cuál es la mejor receta de paella valenciana?
Route: out_of_scope
Reasoning: This query is about a recipe for paella, which is unrelated to Atalaya Data's services or support areas. Therefore, it falls under out_of_scope.
Sections used:
  (none)
Answer: Lo siento, solo puedo responder consultas sobre Atalaya Data relacionadas con recursos humanos, tecnología o finanzas. Tu consulta no corresponde a ninguno de esos temas.
abstained: False
Trace: https://us.cloud.langfuse.com/project/cmv48hrxk012mad0iu5ad20ir/traces/7b9fcaa66497568e136deaaceae5bea0


### Abstention

The Router sees an HR-shaped Query, but no Section reaches the Similarity Threshold, so the HR Specialist abstains. ADR 0004 warns that unanswerable Queries rarely abstain at this threshold; this one is picked because its best Section stays below 0.325.

In [11]:
answer = system.answer_query('¿Cuándo vencen mis stock options?')
print(format_support_answer(answer, system.langfuse.get_trace_url(trace_id=answer.trace_id)))

Query: ¿Cuándo vencen mis stock options?
Route: hr
Reasoning: La consulta sobre el vencimiento de las stock options se relaciona con temas de empleo y beneficios para los empleados de Atalaya, lo que corresponde al área de recursos humanos (hr).
Sections used:
  (none)
Answer: Lo siento, la documentación disponible no cubre tu consulta. Te recomendamos contactar directamente al equipo correspondiente.
abstained: True
Trace: https://us.cloud.langfuse.com/project/cmv48hrxk012mad0iu5ad20ir/traces/e2a06fcb49f3fc5f547d342741185dc0


## 6. Langfuse integration

Every `answer_query` call is one Langfuse trace. The Judge's four scores (`quality`, `relevance`, `completeness`, `accuracy`) and the `abstained` score are published on that trace, so they can be filtered and compared in the Langfuse UI.

### Calibration sweep

The sweep compares Similarities only: no model is called and no trace is created. For each candidate threshold it counts how many answerable calibration Test Queries keep their expected Section and how many unanswerable ones would abstain. Its result is saved in `outputs/threshold_sweep.json` and is the evidence behind ADR 0004.

In [12]:
calibration = load_test_queries("test_queries.json", "calibration")
thresholds = [round(0.25 + 0.025 * i, 3) for i in range(20)]
points = sweep_threshold(system.retrieve_by_area, calibration, thresholds)
Path("outputs").mkdir(exist_ok=True)
Path("outputs/threshold_sweep.json").write_text(json.dumps([p.model_dump() for p in points], indent=2) + "\n")
for p in points:
    print(
        f"threshold {p.threshold:.3f}: retrieval hits {p.retrieval_hits}/{p.answerable}, "
        f"correct abstentions {p.correct_abstentions}/{p.unanswerable}"
    )

threshold 0.250: retrieval hits 24/24, correct abstentions 0/5
threshold 0.275: retrieval hits 24/24, correct abstentions 0/5
threshold 0.300: retrieval hits 24/24, correct abstentions 0/5
threshold 0.325: retrieval hits 24/24, correct abstentions 0/5
threshold 0.350: retrieval hits 24/24, correct abstentions 0/5
threshold 0.375: retrieval hits 24/24, correct abstentions 0/5
threshold 0.400: retrieval hits 24/24, correct abstentions 0/5
threshold 0.425: retrieval hits 23/24, correct abstentions 0/5
threshold 0.450: retrieval hits 22/24, correct abstentions 0/5
threshold 0.475: retrieval hits 22/24, correct abstentions 0/5
threshold 0.500: retrieval hits 21/24, correct abstentions 1/5
threshold 0.525: retrieval hits 19/24, correct abstentions 3/5
threshold 0.550: retrieval hits 16/24, correct abstentions 4/5
threshold 0.575: retrieval hits 12/24, correct abstentions 4/5
threshold 0.600: retrieval hits 8/24, correct abstentions 4/5
threshold 0.625: retrieval hits 6/24, correct abstention

### Validation Experiment

The validation split is uploaded as a Langfuse Dataset and `answer_query` is run on every item as an Experiment. Three Checks score each item (routing correct, retrieval hit, whether the Specialist abstained), next to the Judge's scores. The link below opens this run in Langfuse, and the summary under it shows this run's Check results per Area. This run is not used to tune anything.

The figures from the ticket 08 reference run (a different, earlier run) were 15/16 Queries routed correctly and 11/12 retrieval hits; this run may differ slightly. Two findings to keep in mind when reading the results:

1. The `abstained` flag undercounts declines: the Specialist's prompt makes it decline in text ("No se menciona en las secciones...") even when some Section passed the threshold, so the flag stays false.
2. A high Judge rating does not prove the Route was right: the Judge only checks the Answer against the Sections it received. finance-07 was routed to HR and still got 10/10.

In [13]:
upload_test_queries(
    system.langfuse, "support-routing-validation", load_test_queries("test_queries.json", "validation")
)
result = run_validation_experiment(
    system.langfuse, "support-routing-validation", "notebook-run", system.answer_query
)
print(result.dataset_run_url)
print(format_experiment_summary(result))

https://us.cloud.langfuse.com/project/cmv48hrxk012mad0iu5ad20ir/datasets/cmv4i3rx1007jad0jplfx4dyb/runs/442683c6-7644-401b-bf95-296c99be37c7
hr: 5 Test Queries, routing 5/5, retrieval 4/4, abstentions 0 (of 5 with an Area)
tech: 6 Test Queries, routing 6/6, retrieval 5/5, abstentions 0 (of 6 with an Area)
finance: 3 Test Queries, routing 2/3, retrieval 2/3, abstentions 0 (of 3 with an Area)
out_of_scope: 2 Test Queries, routing 2/2, retrieval —, abstentions —
Total: 16 Test Queries, routing 15/16, retrieval 11/12, abstentions 0 (of 14 with an Area)


In [14]:
system.langfuse.flush()